# EfficientNet-B0 — Notebook final simplifié

Ce notebook garde uniquement la partie utile pour :
- entraîner EfficientNet-B0 en binaire (`no_tumor` vs `tumor`)
- sauvegarder les checkpoints
- comparer `standard` vs `TTA`
- évaluer sur le jeu de test final

Les parties exploration / data understanding ont été retirées.


In [ ]:
!pip install -q torch torchvision torchaudio scikit-learn matplotlib seaborn kagglehub albumentations tqdm


In [ ]:
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
)
from tqdm.auto import tqdm

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
sns.set_palette("husl")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Seed:", SEED)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("sartajbhuvaji/brain-tumor-classification-mri")
)
print("Dataset:", dataset_path)

TRAIN_DIR = dataset_path / "Training"
TEST_DIR = dataset_path / "Testing"

if not TRAIN_DIR.exists():
    for d in dataset_path.rglob("Training"):
        TRAIN_DIR = d
        break

if not TEST_DIR.exists():
    for d in dataset_path.rglob("Testing"):
        TEST_DIR = d
        break

print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

class BrainMRIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dossier introuvable : {self.root_dir}")

        for cls_dir in sorted(self.root_dir.iterdir()):
            if not cls_dir.is_dir():
                continue

            label = 0 if cls_dir.name == "no_tumor" else 1

            for img_path in sorted(cls_dir.iterdir()):
                if img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((img_path, label))

        if not self.samples:
            raise ValueError(f"Aucune image trouvée dans {self.root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image_np = np.array(Image.open(img_path).convert("RGB"))

        if self.transform is not None:
            image = self.transform(image=image_np)["image"]
        else:
            image = image_np

        label = torch.tensor(label, dtype=torch.long)
        return image, label

full_dataset = BrainMRIDataset(TRAIN_DIR, transform=None)
labels_all = [full_dataset.samples[i][1] for i in range(len(full_dataset))]
print("Images train:", len(labels_all))
print("Distribution train:", Counter(labels_all))


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224

def get_train_transforms(augmentation_level="standard"):
    if augmentation_level == "standard":
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Rotate(limit=12, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
            A.CLAHE(clip_limit=2.0, p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.4),
            A.GaussNoise(var_limit=(5.0, 25.0), p=0.2),
            A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.3),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])

def get_eval_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_tta_transforms():
    return [
        get_eval_transforms(),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.VerticalFlip(p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Rotate(limit=(10, 10), p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
        A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Rotate(limit=(-10, -10), p=1.0),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]),
    ]


In [ ]:
idx_all = np.arange(len(labels_all))
train_idx, val_idx = train_test_split(
    idx_all,
    test_size=0.2,
    random_state=SEED,
    stratify=labels_all,
)

train_labels = [labels_all[i] for i in train_idx]
val_labels = [labels_all[i] for i in val_idx]
print("Train:", len(train_idx), Counter(train_labels))
print("Val  :", len(val_idx), Counter(val_labels))

BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = BrainMRIDataset(TRAIN_DIR, transform=get_train_transforms("standard"))
val_dataset = BrainMRIDataset(TRAIN_DIR, transform=get_eval_transforms())

train_set = Subset(train_dataset, train_idx)
val_set = Subset(val_dataset, val_idx)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


In [ ]:
def build_model(num_classes=2, pretrained=True):
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    try:
        model = efficientnet_b0(weights=weights)
        print("EfficientNet-B0 chargé.")
    except Exception as e:
        print("Pré-entraînement indisponible, fallback sans poids:", e)
        model = efficientnet_b0(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

def freeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True
    return model

def unfreeze_last_blocks(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.features[6].parameters():
        p.requires_grad = True
    for p in model.features[7].parameters():
        p.requires_grad = True
    for p in model.features[8].parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True
    return model

model = build_model(num_classes=2, pretrained=True).to(DEVICE)
print("Paramètres totaux:", sum(p.numel() for p in model.parameters()))


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, n_samples, correct = 0.0, 0, 0
    all_preds, all_targets = [], []

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(labels.cpu().tolist())

    return {
        "loss": running_loss / n_samples,
        "acc": correct / n_samples,
        "preds": all_preds,
        "targets": all_targets,
    }

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, n_samples, correct = 0.0, 0, 0
    all_preds, all_targets, all_probs = [], [], []

    for images, labels in tqdm(loader, desc="Val", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        n_samples += images.size(0)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(labels.cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

    return {
        "loss": running_loss / n_samples,
        "acc": correct / n_samples,
        "preds": all_preds,
        "targets": all_targets,
        "probs": all_probs,
    }

@torch.no_grad()
def predict_tta(model, dataset, device, tta_transforms=None):
    if tta_transforms is None:
        tta_transforms = get_tta_transforms()

    model.eval()
    all_probs = []
    all_targets = []

    for idx in tqdm(range(len(dataset)), desc="TTA", leave=False):
        img_path, label = dataset.samples[idx] if hasattr(dataset, "samples") else dataset.dataset.samples[dataset.indices[idx]]
        image_np = np.array(Image.open(img_path).convert("RGB"))

        probs_per_version = []
        for t in tta_transforms:
            img_t = t(image=image_np)["image"].unsqueeze(0).to(device)
            output = model(img_t)
            prob = torch.softmax(output, dim=1)[0, 1].item()
            probs_per_version.append(prob)

        avg_prob = np.mean(probs_per_version)
        all_probs.append(avg_prob)
        all_targets.append(label)

    return {"probs": all_probs, "targets": all_targets}

def predict_with_threshold(probs, threshold=0.3):
    return [1 if p >= threshold else 0 for p in probs]

def save_checkpoint(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)
    print(f"✅ Modèle sauvegardé : {path}")

def load_checkpoint(model, path, map_location=None):
    state_dict = torch.load(path, map_location=map_location, weights_only=True)
    model.load_state_dict(state_dict)
    return model

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "report": classification_report(y_true, y_pred, zero_division=0, digits=4),
    }

THRESHOLD = 0.3
print("Seuil TTA:", THRESHOLD)


In [ ]:
model = freeze_backbone(model)

class_weights = torch.tensor([2475 / 395, 1.0]).to(DEVICE)
criterion_train = nn.CrossEntropyLoss(weight=class_weights)
criterion_eval = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
EPOCHS_A = 15
PATIENCE = 3
scheduler_a = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_A)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
patience_counter = 0

print("Loss pondérée sain =", class_weights[0].item())
print("Params entraînables Phase A =", sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(EPOCHS_A):
    current_lr = optimizer.param_groups[0]["lr"]
    train_r = train_one_epoch(model, train_loader, criterion_train, optimizer, DEVICE)
    val_r = validate_one_epoch(model, val_loader, criterion_eval, DEVICE)
    scheduler_a.step()

    history["train_loss"].append(train_r["loss"])
    history["val_loss"].append(val_r["loss"])
    history["train_acc"].append(train_r["acc"])
    history["val_acc"].append(val_r["acc"])

    if val_r["loss"] < best_val_loss:
        best_val_loss = val_r["loss"]
        patience_counter = 0
        save_checkpoint(model, "artifacts/checkpoints/efficientnet_phase_a.pt")
        marker = "✅ BEST"
    else:
        patience_counter += 1
        marker = f"(patience {patience_counter}/{PATIENCE})"

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS_A} | LR: {current_lr:.6f} | "
        f"Train Loss: {train_r['loss']:.4f} Acc: {train_r['acc']:.4f} | "
        f"Val Loss: {val_r['loss']:.4f} Acc: {val_r['acc']:.4f} {marker}"
    )

    if patience_counter >= PATIENCE:
        print("⏹ Early stopping Phase A")
        break

actual_epochs_a = len(history["train_loss"])
print("Phase A terminée. Epochs:", actual_epochs_a)


In [ ]:
model = build_model(num_classes=2, pretrained=False).to(DEVICE)
model = load_checkpoint(model, "artifacts/checkpoints/efficientnet_phase_a.pt", map_location=DEVICE)
model = unfreeze_last_blocks(model)

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
EPOCHS_B = 25
PATIENCE = 5
scheduler_b = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_B)

train_dataset.transform = get_train_transforms("advanced")

best_val_loss_b = float("inf")
best_val_acc = 0.0
patience_counter = 0

print("Params entraînables Phase B =", sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(EPOCHS_B):
    current_lr = optimizer.param_groups[0]["lr"]
    train_r = train_one_epoch(model, train_loader, criterion_train, optimizer, DEVICE)
    val_r = validate_one_epoch(model, val_loader, criterion_eval, DEVICE)
    scheduler_b.step()

    history["train_loss"].append(train_r["loss"])
    history["val_loss"].append(val_r["loss"])
    history["train_acc"].append(train_r["acc"])
    history["val_acc"].append(val_r["acc"])

    if val_r["loss"] < best_val_loss_b:
        best_val_loss_b = val_r["loss"]
        patience_counter = 0
        save_checkpoint(model, "artifacts/checkpoints/efficientnet_best.pt")
        marker = "✅ BEST"
    else:
        patience_counter += 1
        marker = f"(patience {patience_counter}/{PATIENCE})"

    if val_r["acc"] > best_val_acc:
        best_val_acc = val_r["acc"]

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS_B} | LR: {current_lr:.6f} | "
        f"Train Loss: {train_r['loss']:.4f} Acc: {train_r['acc']:.4f} | "
        f"Val Loss: {val_r['loss']:.4f} Acc: {val_r['acc']:.4f} {marker}"
    )

    if patience_counter >= PATIENCE:
        print("⏹ Early stopping Phase B")
        break

actual_epochs_b = len(history["train_loss"]) - actual_epochs_a
print("Phase B terminée. Epochs:", actual_epochs_b, "| Best val acc:", best_val_acc)


In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(epochs_range, history["train_loss"], "b-o", label="Train", linewidth=2, markersize=4)
ax1.plot(epochs_range, history["val_loss"], "r-o", label="Validation", linewidth=2, markersize=4)
ax1.axvline(x=actual_epochs_a, color="gray", linestyle="--", alpha=0.7, label="Transition")
ax1.set_title("Loss par époque")
ax1.set_xlabel("Époque")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], "b-o", label="Train", linewidth=2, markersize=4)
ax2.plot(epochs_range, history["val_acc"], "r-o", label="Validation", linewidth=2, markersize=4)
ax2.axvline(x=actual_epochs_a, color="gray", linestyle="--", alpha=0.7, label="Transition")
ax2.set_title("Accuracy par époque")
ax2.set_xlabel("Époque")
ax2.set_ylabel("Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
best_model = build_model(num_classes=2, pretrained=False).to(DEVICE)
best_model = load_checkpoint(best_model, "artifacts/checkpoints/efficientnet_best.pt", map_location=DEVICE)

val_result = validate_one_epoch(best_model, val_loader, criterion_eval, DEVICE)
val_tta = predict_tta(best_model, val_set, DEVICE)

pred_std050 = predict_with_threshold(val_result["probs"], threshold=0.5)
pred_std030 = predict_with_threshold(val_result["probs"], threshold=THRESHOLD)
pred_tta030 = predict_with_threshold(val_tta["probs"], threshold=THRESHOLD)

m_std050 = compute_metrics(val_result["targets"], pred_std050)
m_std030 = compute_metrics(val_result["targets"], pred_std030)
m_tta030 = compute_metrics(val_tta["targets"], pred_tta030)

print("=== COMPARAISON VALIDATION ===")
print("Std 0.5 | acc={:.4f} precision={:.4f} recall={:.4f} f1={:.4f}".format(
    m_std050["accuracy"], m_std050["precision"], m_std050["recall"], m_std050["f1"]
))
print("Std 0.3 | acc={:.4f} precision={:.4f} recall={:.4f} f1={:.4f}".format(
    m_std030["accuracy"], m_std030["precision"], m_std030["recall"], m_std030["f1"]
))
print("TTA 0.3 | acc={:.4f} precision={:.4f} recall={:.4f} f1={:.4f}".format(
    m_tta030["accuracy"], m_tta030["precision"], m_tta030["recall"], m_tta030["f1"]
))
print(m_tta030["report"])


In [ ]:
cm = m_tta030["confusion_matrix"]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    linewidths=1,
    xticklabels=["Sain (0)", "Tumeur (1)"],
    yticklabels=["Sain (0)", "Tumeur (1)"],
    ax=ax,
)
ax.set_xlabel("Prédiction")
ax.set_ylabel("Réalité")
ax.set_title("Matrice de confusion — validation TTA seuil 0.3")
plt.show()

tn, fp, fn, tp = cm.ravel()
print("TN =", tn, "| FP =", fp, "| FN =", fn, "| TP =", tp)


In [ ]:
test_dataset = BrainMRIDataset(TEST_DIR, transform=get_eval_transforms())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_labels = [test_dataset.samples[i][1] for i in range(len(test_dataset))]
print("Test size:", len(test_dataset))
print("Test distribution:", Counter(test_labels))

std_test = validate_one_epoch(best_model, test_loader, criterion_eval, DEVICE)
tta_test = predict_tta(best_model, test_dataset, DEVICE)

t_std050 = predict_with_threshold(std_test["probs"], threshold=0.5)
t_std030 = predict_with_threshold(std_test["probs"], threshold=THRESHOLD)
t_tta030 = predict_with_threshold(tta_test["probs"], threshold=THRESHOLD)

tm_std050 = compute_metrics(std_test["targets"], t_std050)
tm_std030 = compute_metrics(std_test["targets"], t_std030)
tm_tta030 = compute_metrics(tta_test["targets"], t_tta030)

print("=== RÉSULTATS TEST FINAL ===")
print("Std 0.5 | acc={:.4f} precision={:.4f} recall={:.4f} f1={:.4f}".format(
    tm_std050["accuracy"], tm_std050["precision"], tm_std050["recall"], tm_std050["f1"]
))
print("Std 0.3 | acc={:.4f} precision={:.4f} recall={:.4f} f1={:.4f}".format(
    tm_std030["accuracy"], tm_std030["precision"], tm_std030["recall"], tm_std030["f1"]
))
print("TTA 0.3 | acc={:.4f} precision={:.4f} recall={:.4f} f1={:.4f}".format(
    tm_tta030["accuracy"], tm_tta030["precision"], tm_tta030["recall"], tm_tta030["f1"]
))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (title, metrics_dict) in zip(
    axes,
    [
        ("Standard seuil 0.5", tm_std050),
        ("Standard seuil 0.3", tm_std030),
        ("TTA seuil 0.3", tm_tta030),
    ],
):
    sns.heatmap(
        metrics_dict["confusion_matrix"],
        annot=True,
        fmt="d",
        cmap="RdYlGn",
        square=True,
        linewidths=1,
        xticklabels=["Sain", "Tumeur"],
        yticklabels=["Sain", "Tumeur"],
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Réalité")

plt.tight_layout()
plt.show()

tn, fp, fn, tp = tm_tta030["confusion_matrix"].ravel()
print("=== TTA seuil 0.3 ===")
print("Tumeurs détectées :", tp, "/", tp + fn)
print("Tumeurs ratées    :", fn)
print("Faux positifs     :", fp)
print("Test set          :", len(test_dataset))


In [ ]:
final_ckpt = Path("artifacts/checkpoints/best_efficientnet_b0.pt")
save_checkpoint(best_model, final_ckpt)
print("Checkpoint final prêt pour le déploiement :", final_ckpt.resolve())
